```
<10_imbalanced_data.ipynb>

제미나이 의존도: 60-70%

파일 이름 그대로 불균형한 데이터를 다루었다.

불균형한 데이터를 어디서 또 가져올지 고민하다가 그냥 기존 과일 데이터를 불균형하게 만들기로 했다.
사과가 약 6000개, 바나나가 약 6000개, 오렌지가 약 4000개 있는데, 여기서 오렌지를 100개로 줄여서 의도적으로 불균형으로 만들어내는 시도를 해보았다.

우선 Train에만 데이터 불균형을 주고 Val과 Test에는 불균형을 주면 안된다.
Val과 Test에도 데이터 불균형을 주게 되면 전부 사과 또는 바나나로 예측해도 정확도가 높게 나오기 때문에 객관적이지 않기 때문이다.
그래서 불균형이 없는 상태에서 Test와 Val을 격리해 낸다.

0. 먼저 전체 데이터프레임을 origin_df로 정의한다.
1. origin_df에서 Test를 10% 분리하여 떼어낸다. except_test_df와 test_df가 생긴다.
2. except_test_df에서 Val을 10% 분리하여 떼어낸다. except_val_df와 val_df가 생긴다.
3. except_val_df에서 오렌지 100개를 솎아내어 train_df를 만든다.

train_df가 준비됐다면, WeightedRandomSampler를 사용해 클래스마다 가중치를 부여한다.
train_df['label'].value_counts()를 통해 클래스별 개수를 구하고, 그 개수에 역수를 취한것을 가중치로 택한다.
즉 개수가 많을수록 역수를 취하면 값이 작아지니 가중치도 작어진다는 뜻이다.

WeightedRandomSampler는 많은 데이터는 적게 뽑고, 적은 데이터는 많이 뽑아서 최대한 균등하게 뽑는 게 목적이기 때문이다.

결과는 나쁘지 않았다.
Epoch 1/5 | Train Loss: 0.1299 | lr: 0.00090460
Val Acc: 95.51% | Val Loss: 0.1980
Epoch 2/5 | Train Loss: 0.0448 | lr: 0.00065485
Val Acc: 95.05% | Val Loss: 0.1990
Epoch 3/5 | Train Loss: 0.0195 | lr: 0.00034615
Val Acc: 97.89% | Val Loss: 0.1008
Epoch 4/5 | Train Loss: 0.0037 | lr: 0.00009640
Val Acc: 97.76% | Val Loss: 0.1145
Epoch 5/5 | Train Loss: 0.0013 | lr: 0.00000100
Val Acc: 97.36% | Val Loss: 0.1313
Test Acc: 97.09%

Val과 Test에 불균형이 없다고 "잘 나왔겠지"라고 착각하면 안된다.
Confusion Matrix와 Classification Report로 불균형이 있는 데이터(4, 5번)도 잘 맞혔는지 확인해 본다.


=== Confusion Matrix ===
[[282   0   0   0   0   0]
 [  1 384   0   0   0   0]
 [  1   0 284   0   0   0]
 [  0   0   0 346   0   0]
 [  8   1   5   0 167   5]
 [  1  16   0   8   3 172]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       282
           1       0.96      1.00      0.98       385
           2       0.98      1.00      0.99       285
           3       0.98      1.00      0.99       346
           4       0.98      0.90      0.94       186
           5       0.97      0.86      0.91       200

    accuracy                           0.97      1684
   macro avg       0.97      0.96      0.96      1684
weighted avg       0.97      0.97      0.97      1684
```

1. origin_df에서 Test 떼어냄
2. 떼어내고 남은 df에서 Val 떼어냄
3. 떼어내고 남은 df에서 오렌지 100개 솎아냄

In [49]:
import os
from glob import glob
import pandas as pd


file_list = glob("images/kaggle_data/dataset/*/*/*.png")

print(len(file_list)) # 16831

data = []
for i, filename in enumerate(file_list):
    label = -1
    
    if "apple/fresh_apple" in filename:
        label = 0
    if "apple/rotten_apple" in filename:
        label = 1
    if "banana/fresh_banana" in filename:
        label = 2
    if "banana/rotten_banana" in filename:
        label = 3
    if "orange/fresh_orange" in filename:
        label = 4
    if "orange/rotten_orange" in filename:
        label = 5

    data.append({
        "filename": filename, 
        "label": label
    })

origin_df = pd.DataFrame(data)
print(f"len of origin_df: {len(origin_df)}")

16831
len of origin_df: 16831


In [63]:
from sklearn.model_selection import train_test_split

# 1. origin_df에서 Test 떼어냄
except_test_df, test_df = train_test_split(
    origin_df, test_size=0.1, stratify=origin_df['label'], random_state=42, shuffle=True
)

except_test_df = except_test_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 2. 떼어내고 남은 df에서 Val 떼어냄
except_val_df, val_df = train_test_split(
    except_test_df, test_size=0.1, stratify=except_test_df['label'], random_state=42, shuffle=True
)

except_val_df = except_val_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(len(except_val_df))

# 3. 떼어내고 남은 df에서 오렌지 100개 솎아냄
orange_df = except_val_df[except_val_df['label'].isin([4, 5])]
print(len(orange_df))
sample_100_orange_df = orange_df.sample(n=100)
print(len(sample_100_orange_df))

fresh_orange_df = sample_100_orange_df[sample_100_orange_df['label'] == 4]
rotten_orange_df = sample_100_orange_df[sample_100_orange_df['label'] == 5]
print(f"len_of_fresh_orange: {len(fresh_orange_df)}, len_of_rotten_orange: {len(rotten_orange_df)}")


except_orange_df = except_val_df[~except_val_df['label'].isin([4, 5])]
print(len(except_orange_df))

train_df = pd.concat([except_orange_df, sample_100_orange_df], ignore_index=True)
print(len(train_df))

print(f"len of train: {len(train_df)}, len of val: {len(val_df)}, len of test: {len(test_df)}")

13632
3119
100
len_of_fresh_orange: 47, len_of_rotten_orange: 53
10513
10613
len of train: 10613, len of val: 1515, len of test: 1684


In [105]:
from torch.utils.data import Dataset, DataLoader
import cv2
import torch

class FruitsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = row['filename']
        image = cv2.imread(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=image)
            image = aug['image']
        
        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long)

In [101]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(128, 128), 
    A.HorizontalFlip(p=0.5), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

In [88]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

# 클래스별 개수 구하기
class_counts = train_df['label'].value_counts()
print(len(class_counts))
print(f"type: {type(class_counts)}")

print(class_counts)

# 가중치 계산하기
weight = 1.0 / class_counts

print(weight)

weight_dict = weight.to_dict()
print(weight_dict)

train_df['sample_weight'] = train_df['label'].map(weight_dict)

orange_df = train_df[train_df['label'].isin([4, 5])]
except_orange_df = train_df[~train_df['label'].isin([4, 5])]

new_df = pd.concat([orange_df.head(), except_orange_df.head()], ignore_index=True)
print(len(new_df))
print(new_df.head(10))


# 샘플별 가중치 리스트 만들기
sample_weights = train_df['sample_weight'].tolist()
# print(f"type: {type(sample_weights)}, len: {len(sample_weights)}")

sampler = WeightedRandomSampler(
    weights=sample_weights, # 샘플별 가중치 리스트
    num_samples=len(sample_weights), # 한 에폭 동안 뽑을 총 샘플 수
    replacement=True # 중복 추출 허용 (소수 클래스를 복사해서 자주 보여줘야 하므로 필수입니다.)
)

6
type: <class 'pandas.Series'>
label
1    3118
3    2804
2    2308
0    2283
5      53
4      47
Name: count, dtype: int64
label
1    0.000321
3    0.000357
2    0.000433
0    0.000438
5    0.018868
4    0.021277
Name: count, dtype: float64
{1: 0.00032071840923669016, 3: 0.0003566333808844508, 2: 0.0004332755632582322, 0: 0.0004380201489268506, 5: 0.018867924528301886, 4: 0.02127659574468085}
10
                                            filename  label  sample_weight
0  images/kaggle_data/dataset/orange/rotten_orang...      5       0.018868
1  images/kaggle_data/dataset/orange/rotten_orang...      5       0.018868
2  images/kaggle_data/dataset/orange/fresh_orange...      4       0.021277
3  images/kaggle_data/dataset/orange/rotten_orang...      5       0.018868
4  images/kaggle_data/dataset/orange/fresh_orange...      4       0.021277
5  images/kaggle_data/dataset/apple/rotten_apple/...      1       0.000321
6  images/kaggle_data/dataset/banana/rotten_banan...      3       0.000357


In [106]:
BATCH_SIZE = 64

train_dataset = FruitsDataset(df=train_df, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)

val_dataset = FruitsDataset(df=val_df, transform=val_test_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataset = FruitsDataset(df=test_df, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [113]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels)

unique_labels, counts = torch.unique(labels, return_counts=True)
print(f"어떤 라벨이 있나? {unique_labels}")
print(f"각각 몇개씩 있나? {counts}")

from collections import Counter
label_counts = Counter(labels.tolist())
print(label_counts)

count_0 = 0
count_1 = 0
count_2 = 0
count_3 = 0
count_4 = 0
count_5 = 0
for label in labels:
    if label.item() == 0:
        count_0 += 1
    elif label.item() == 1:
        count_1 += 1
    elif label.item() == 2:
        count_2 += 1
    elif label.item() == 3:
        count_3 += 1
    elif label.item() == 4:
        count_4 += 1
    elif label.item() == 5:
        count_5 += 1

print(f"0: {count_0}, 1: {count_1}, 2: {count_2}, 3: {count_3}, 4: {count_4}, 5: {count_5}")


torch.Size([64, 3, 128, 128])
torch.Size([64])
tensor([5, 5, 0, 1, 3, 1, 0, 4, 1, 4, 4, 4, 5, 0, 4, 4, 1, 4, 4, 5, 5, 3, 3, 0,
        5, 2, 5, 2, 2, 3, 0, 3, 0, 4, 0, 1, 4, 3, 4, 5, 3, 3, 1, 2, 5, 0, 2, 3,
        1, 5, 2, 2, 0, 5, 4, 0, 1, 2, 4, 0, 1, 2, 4, 5])
어떤 라벨이 있나? tensor([0, 1, 2, 3, 4, 5])
각각 몇개씩 있나? tensor([11,  9,  9,  9, 14, 12])
Counter({4: 14, 5: 12, 0: 11, 1: 9, 3: 9, 2: 9})
0: 11, 1: 9, 2: 9, 3: 9, 4: 14, 5: 12


In [129]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 5
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"device: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

device: mps


In [130]:
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # === Train Start ===
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_dataset):.4f} | lr: {current_lr:.8f}")
    # === Train End ===
    
    # === Val Start ===
    model.eval()
    with torch.no_grad():
        val_correct = 0
        val_loss = 0.0
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
    
            outputs = model(images)
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
    
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
    
    val_loss /= len(val_dataset)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "imbalanced_data_best_model.pth")
    
    print(f"Val Acc: {val_correct/len(val_dataset) * 100:.2f}% | Val Loss: {val_loss:.4f}")
    # === Val End ===
    
# === Test Start ===
best_model = models.resnet18(weights=None)
best_model.fc = nn.Linear(best_model.fc.in_features, 6)
best_model.load_state_dict(torch.load("imbalanced_data_best_model.pth"))
best_model = best_model.to(device)
best_model.eval()

test_correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = best_model(images)
        preds = outputs.argmax(1)
        test_correct += (preds == labels).sum().item()

print(f"Test Acc: {test_correct/len(test_dataset) * 100:.2f}%")        
# === Test End ===

Epoch 1/5 | Train Loss: 0.1299 | lr: 0.00090460
Val Acc: 95.51% | Val Loss: 0.1980
Epoch 2/5 | Train Loss: 0.0448 | lr: 0.00065485
Val Acc: 95.05% | Val Loss: 0.1990
Epoch 3/5 | Train Loss: 0.0195 | lr: 0.00034615
Val Acc: 97.89% | Val Loss: 0.1008
Epoch 4/5 | Train Loss: 0.0037 | lr: 0.00009640
Val Acc: 97.76% | Val Loss: 0.1145
Epoch 5/5 | Train Loss: 0.0013 | lr: 0.00000100
Val Acc: 97.36% | Val Loss: 0.1313
Test Acc: 97.09%


In [131]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = models.resnet18(weights=None)
best_model.fc = nn.Linear(best_model.fc.in_features, 6)
best_model.load_state_dict(torch.load("imbalanced_data_best_model.pth"))
best_model = best_model.to(device)
best_model.eval()


true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = best_model(images)
        preds = outputs.argmax(1)

        labels = labels.cpu().numpy().tolist()
        preds = preds.cpu().numpy().tolist()
        true_labels.extend(labels)
        pred_labels.extend(preds)
        
        # test_correct += (preds == labels).sum().item()


In [132]:
print("=== Confusion Matrix ===")
print(confusion_matrix(true_labels, pred_labels))
print("\n=== Classification Report ===")
print(classification_report(true_labels, pred_labels))


=== Confusion Matrix ===
[[282   0   0   0   0   0]
 [  1 384   0   0   0   0]
 [  1   0 284   0   0   0]
 [  0   0   0 346   0   0]
 [  8   1   5   0 167   5]
 [  1  16   0   8   3 172]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       282
           1       0.96      1.00      0.98       385
           2       0.98      1.00      0.99       285
           3       0.98      1.00      0.99       346
           4       0.98      0.90      0.94       186
           5       0.97      0.86      0.91       200

    accuracy                           0.97      1684
   macro avg       0.97      0.96      0.96      1684
weighted avg       0.97      0.97      0.97      1684

